# Modelos Baseline — Random Forest y SVM

**Objetivo:** Clasificar actividad humana usando modelos clásicos de ML.

Como WISDM provee señal cruda (no features precalculadas como UCI HAR),  
el primer paso es **extraer features estadísticas** manualmente de cada ventana.

Pipeline:
```
X_raw (n, 90, 3) → features estadísticas (n, 33) → StandardScaler → RF / SVM
```

Las 33 features son: 11 estadísticos × 3 ejes (mean, std, mad, max, min, range, energy, iqr, skewness, kurtosis, rms)

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from pathlib import Path
from IPython.display import display

from src.data_loader import load_dataset, load_activity_labels
from src.features import extract_features_from_windows, get_feature_names, normalize_signals
from src.evaluate import evaluate_model, plot_confusion_matrix
from src.models.baseline import train_random_forest, train_svm

FIGURES_DIR = Path('..') / 'figures'
MODELS_DIR  = Path('..') / 'models'
FIGURES_DIR.mkdir(exist_ok=True)
MODELS_DIR.mkdir(exist_ok=True)

plt.style.use('seaborn-v0_8-whitegrid')
print('Setup OK')

## 1. Carga y transformación de datos

**¿Por qué extraer features estadísticas?**  
Random Forest y SVM esperan una matriz 2D (n_samples, n_features).  
Las señales crudas tienen forma (n_samples, 90, 3) — necesitamos aplanarlas  
de forma significativa, no simplemente concatenar los 270 valores brutos.

In [ ]:
X_train_raw, y_train = load_dataset('train')
X_test_raw,  y_test  = load_dataset('test')
labels = load_activity_labels()

print('Extrayendo features estadísticas...')
X_train_feats = extract_features_from_windows(X_train_raw)
X_test_feats  = extract_features_from_windows(X_test_raw)
feature_names = get_feature_names()

print(f'Señal cruda  → {X_train_raw.shape}')
print(f'Features     → {X_train_feats.shape}  (33 = 11 estadísticos × 3 ejes)')
print(f'\nPrimeras features: {feature_names[:6]}')

In [ ]:
# Normalización: StandardScaler ajustado SOLO en train, aplicado a test
# Esto evita data leakage: el test no puede influir en la normalización
X_train, X_test, scaler = normalize_signals(X_train_feats, X_test_feats)

print('Normalización aplicada (media 0, std 1 por feature).')
print(f'Media train post-normalización: {X_train.mean():.6f}  (debe ser ~0)')
print(f'Std  train post-normalización: {X_train.std():.6f}   (debe ser ~1)')

# Guardar scaler para usarlo en evaluación
joblib.dump(scaler, MODELS_DIR / 'scaler.joblib')
print('Scaler guardado.')

## 2. Random Forest

**¿Por qué Random Forest?**
- Maneja bien datasets medianos sin mucho tuning
- Robusto al ruido y outlers
- Provee feature importance interpretable
- No requiere normalización (aunque la aplicamos igual para ser consistentes)

Usamos 200 árboles — suficientes para estabilizar la varianza del ensemble.

In [ ]:
print('Entrenando Random Forest (200 árboles, n_jobs=-1)...')
rf = train_random_forest(X_train, y_train, n_estimators=200)
y_pred_rf = rf.predict(X_test)

metrics_rf = evaluate_model(y_test, y_pred_rf, labels, 'Random Forest')

# Guardar predicciones y modelo
np.save(MODELS_DIR / 'rf_preds.npy', y_pred_rf)
joblib.dump(rf, MODELS_DIR / 'random_forest.joblib')
print('Modelo y predicciones guardados.')

In [ ]:
plot_confusion_matrix(y_test, y_pred_rf, labels, 'Random Forest')

### Feature Importance — Random Forest

Random Forest puede decirnos cuáles de las 33 features son más útiles para clasificar.  
Esto es valioso para **interpretar el modelo** — el profesor puede preguntar "¿qué feature es más importante y por qué?"

In [ ]:
from src.features import feature_importance_rf

top_feats = feature_importance_rf(rf.feature_importances_, feature_names, top_n=15)
names, imps = zip(*top_feats)

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(list(names)[::-1], list(imps)[::-1], color='steelblue')
ax.set_xlabel('Importancia (Gini impurity)')
ax.set_title('Top 15 features — Random Forest')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'rf_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print('Top 5 features más importantes:')
for name, imp in top_feats[:5]:
    print(f'  {name:25s}: {imp:.4f}')

## 3. SVM — LinearSVC

**¿Por qué LinearSVC en lugar de SVC(kernel='rbf')?**  
Con 43.924 muestras de entrenamiento, SVM con kernel RBF tardaría horas  
(complejidad O(n²) a O(n³)). LinearSVC es O(n × features) y funciona  
muy bien cuando las features están normalizadas, que es exactamente nuestro caso.

El SVM busca el **hiperplano de máximo margen** que separa las clases.  
C=0.1 significa regularización fuerte (márgenes más amplios, menos overfitting).

In [ ]:
print('Entrenando SVM (LinearSVC, C=0.1)...')
svm = train_svm(X_train, y_train, C=0.1)
y_pred_svm = svm.predict(X_test)

metrics_svm = evaluate_model(y_test, y_pred_svm, labels, 'SVM (LinearSVC)')

np.save(MODELS_DIR / 'svm_preds.npy', y_pred_svm)
joblib.dump(svm, MODELS_DIR / 'svm.joblib')
print('Modelo y predicciones guardados.')

In [ ]:
plot_confusion_matrix(y_test, y_pred_svm, labels, 'SVM LinearSVC')

## 4. Comparativa de modelos clásicos

In [ ]:
from sklearn.metrics import f1_score

results = pd.DataFrame([
    {'Modelo': 'Random Forest', **metrics_rf},
    {'Modelo': 'SVM (LinearSVC)', **metrics_svm},
]).set_index('Modelo')

print('=== RESULTADOS MODELOS CLÁSICOS ===')
display(results.round(4).style.highlight_max(color='lightgreen', axis=0))

# F1 por clase
label_names = [labels[k] for k in sorted(labels)]
f1_rf  = f1_score(y_test, y_pred_rf,  average=None)
f1_svm = f1_score(y_test, y_pred_svm, average=None)

df_f1 = pd.DataFrame({'Random Forest': f1_rf, 'SVM': f1_svm}, index=label_names)
print('\nF1-score por actividad:')
display(df_f1.round(4).style.background_gradient(cmap='RdYlGn', vmin=0, vmax=1))

## Conclusiones — Modelos Clásicos

- **Random Forest** y **SVM** logran buen rendimiento con solo 33 features estadísticas.
- Las clases más difíciles son **Sitting vs Standing** (señal casi idéntica en el dominio estadístico).
- La **normalización** es crítica para el SVM (sin ella, features con mayor escala dominarían el margen).
- **Próximo paso:** ¿Puede Deep Learning, al ver la señal cruda completa, aprender a distinguir mejor Sitting/Standing?

**Para defender ante el profesor:**
- ¿Por qué 33 features y no concatenar los 270 valores brutos? → Las features estadísticas capturan propiedades invariantes a la fase; 270 valores raw son muy ruidosos para RF/SVM.
- ¿Por qué no ajustar hiperparámetros con GridSearchCV? → Con este tamaño de dataset y 3-fold CV, el costo computacional es alto. Los valores por defecto + exploración son suficientes para el objetivo pedagógico.